# Лабораторная работа №6

## Нейро-символьная система контроля дефектов закупорки молочной тары

Ноутбук предназначен для реального запуска системы через `YandexGPT`.

Перед выполнением необходимо задать в `.env`:
- `YANDEX_IAM_TOKEN`
- `YANDEX_FOLDER_ID`

Далее можно проверить, как модель и символьные правила работают на нескольких производственных ситуациях.

In [1]:
from pprint import pprint

from src.neuro_symbolic.pipeline import NeuroSymbolicPipeline


## 1. Инициализация пайплайна

In [2]:
pipeline = NeuroSymbolicPipeline()
pprint(pipeline.get_statistics())


{'domain': 'milk_packaging_quality',
 'knowledge_base': {'avg_confidence': 1.0, 'subjects': 1, 'total_facts': 6},
 'rules': {'rules_by_domain': {'milk_packaging_quality': 7},
           'rules_by_priority': {'CRITICAL': 1,
                                 'HIGH': 3,
                                 'LOW': 1,
                                 'MEDIUM': 2},
           'total_inferences': 0,
           'total_rules': 7},
 'weights': {'neural': 0.55, 'symbolic': 0.45}}


## 2. Набор реальных производственных сценариев

In [3]:
scenarios = [
    {
        "name": "Критический перекос крышки",
        "query": "На линии вырос процент дефектов закупорки, наблюдается перекос крышки и падение герметичности.",
        "facts": {
            "cap_alignment_error_mm": 2.4,
            "seal_integrity_score": 0.72,
            "cap_torque_nm": 1.55,
            "vision_confidence": 0.68,
            "defect_rate_percent": 4.2,
            "conveyor_speed_bpm": 196,
        },
        "categories": ["норма", "предупреждение", "критично"],
    },
    {
        "name": "Пограничное предупреждение",
        "query": "Есть отклонения по моменту закрутки, снизилась уверенность визуального контроля и начал расти процент брака.",
        "facts": {
            "cap_alignment_error_mm": 1.2,
            "seal_integrity_score": 0.91,
            "cap_torque_nm": 0.88,
            "vision_confidence": 0.74,
            "defect_rate_percent": 1.6,
            "conveyor_speed_bpm": 178,
        },
        "categories": ["норма", "предупреждение", "критично"],
    },
    {
        "name": "Рост брака из-за скорости линии",
        "query": "После увеличения скорости конвейера возросла доля дефектной тары, хотя герметичность пока близка к норме.",
        "facts": {
            "cap_alignment_error_mm": 0.9,
            "seal_integrity_score": 0.94,
            "cap_torque_nm": 1.32,
            "vision_confidence": 0.89,
            "defect_rate_percent": 3.4,
            "conveyor_speed_bpm": 188,
        },
        "categories": ["норма", "предупреждение", "критично"],
    },
    {
        "name": "Штатная работа линии",
        "query": "Линия укупорки работает стабильно, критичных отклонений не наблюдается.",
        "facts": {
            "cap_alignment_error_mm": 0.4,
            "seal_integrity_score": 0.98,
            "cap_torque_nm": 1.10,
            "vision_confidence": 0.96,
            "defect_rate_percent": 0.4,
            "conveyor_speed_bpm": 150,
        },
        "categories": ["норма", "предупреждение", "критично"],
    },
]


## 3. Запуск сценариев через YandexGPT и Rule Engine

In [4]:
results = []

for scenario in scenarios:
    result = pipeline.process(
        {
            "query": scenario["query"],
            "facts": scenario["facts"],
            "categories": scenario["categories"],
        }
    )
    results.append((scenario["name"], result))

    print("=" * 100)
    print("Сценарий:", scenario["name"])
    print("=" * 100)
    print("Факты:")
    pprint(scenario["facts"])
    print("\nФинальное решение:")
    print(result["final_decision"])
    print("\nУверенность:", result["confidence"])
    print("\nКатегория LLM:", result["neural_output"]["classification"]["predicted_category"])
    print("\nLLM-анализ:")
    print(result["neural_output"]["analysis"])
    print("\nСимвольные выводы:")
    for conclusion in result["symbolic_output"]["conclusions"]:
        print("-", conclusion)
    print("\nОбъяснение:")
    print(result["explanation"])
    print()


Сценарий: Критический перекос крышки
Факты:
{'cap_alignment_error_mm': 2.4,
 'cap_torque_nm': 1.55,
 'conveyor_speed_bpm': 196,
 'defect_rate_percent': 4.2,
 'seal_integrity_score': 0.72,
 'vision_confidence': 0.68}

Финальное решение:
Категория LLM: Критично. Нейронный анализ: Состояние линии неудовлетворительное, высок риск появления дефектов.

Проблемы:
* Перекос крышки превышает допустимые значения (2,4 мм при норме 1,0 мм).
* Герметичность снижена (оценка 0,72 при норме 0,95).
* Момент затяжки крышки выходит за пределы нормы (1,55 Н·м при допустимом диапазоне 0,9–1,4 Н·м).
* Доверительный уровень системы компьютерного зрения ниже требуемого (0,68 при норме 0,9).
* Процент дефектов превышает норму (4,2 % при максимально допустимом уровне 1 %).
* Скорость конвейера выше установленной (196 BPM при норме 180 BPM). Символьные выводы: Нейронный анализ изображения недостаточно надёжен для автоматического решения.; Обнаружен критический дефект позиционирования крышки.; Есть риск нарушения

## 4. Сводная таблица результатов

In [5]:
summary = []

for name, result in results:
    summary.append(
        {
            "scenario": name,
            "confidence": result["confidence"],
            "classification": result["neural_output"]["classification"]["predicted_category"],
            "rules_triggered": len(result["symbolic_output"]["triggered_rules"]),
        }
    )

pprint(summary)


[{'classification': 'Критично',
  'confidence': 0.917,
  'rules_triggered': 6,
  'scenario': 'Критический перекос крышки'},
 {'classification': 'предупреждение',
  'confidence': 0.917,
  'rules_triggered': 2,
  'scenario': 'Пограничное предупреждение'},
 {'classification': 'Предупреждение',
  'confidence': 0.917,
  'rules_triggered': 2,
  'scenario': 'Рост брака из-за скорости линии'},
 {'classification': 'норма',
  'confidence': 0.917,
  'rules_triggered': 1,
  'scenario': 'Штатная работа линии'}]
